<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 6**
The code below gets a dataset to predict default. The outcome variable of interest is `default.payment.next.month`. Use two traditional machine learning algorithms (random forest, XGboost, etc.) and TabPFN to predict the outcome. Use a test set of 25% of the data. How well does TabPFN perform in comparison to machine learning algorithms?

In [1]:
import kagglehub
import pandas as pd
import os

dataset_path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")

# List contents of the downloaded dataset directory to find the data file
files_in_dataset = os.listdir(dataset_path)
print(f"Files in the dataset directory: {files_in_dataset}")

# Assuming the main data file is a CSV and is directly in the downloaded path
# You might need to adjust the filename if it's different or in a subdirectory
# For this dataset, it's typically 'UCI_Credit_Card.csv'

# Construct the full path to the CSV file
csv_file_name = 'UCI_Credit_Card.csv'
full_csv_path = os.path.join(dataset_path, csv_file_name)

# Load the CSV into a pandas DataFrame
df = pd.read_csv(full_csv_path)

print("Data loaded into DataFrame 'df' successfully.")
print(df.head())

100%|██████████| 0.98M/0.98M [00:01<00:00, 818kB/s]

Extracting files...


Files in the dataset directory: ['UCI_Credit_Card.csv']
Data loaded into DataFrame 'df' successfully.
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0       0.0   
1  ...     3272.0     3455.0     3261.0       0.0    1000.0    1000.0   
2  ...    14331.0    14948.0    15549.0    1518.0    1500.0    1000.0   
3  ...    28314.0    28959.0    29547.0    2000.0    2019.0    1200.0   
4  ...    20940.0    19146.0    19131.0    2000.0   36681.

Check the info of the dataset

In [3]:
# Basic overview
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

# Check data types
print("\nData types of each column:")
print(df.dtypes)

# Count columns by dtype
print("\nCount of columns by dtype:")
print(df.dtypes.value_counts())

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Check whether there are any non-numeric columns
non_numeric_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
print("\nNon-numeric columns:")
print(non_numeric_cols)

# Show unique values for non-numeric columns if any
if non_numeric_cols:
    print("\nUnique values in non-numeric columns:")
    for col in non_numeric_cols:
        print(f"{col}: {df[col].unique()[:10]}")  # show first 10 unique values
else:
    print("\nAll columns are numeric.")

# identify target column and inspect it
target_col = "default.payment.next.month"   # target in this dataset

if target_col in df.columns:
    print(f"\nTarget column: {target_col}")
    print("Target dtype:", df[target_col].dtype)
    print("Target unique values:", df[target_col].unique())
    print("Target value counts:")
    print(df[target_col].value_counts())
else:
    print(f"\nTarget column '{target_col}' not found. Please verify the column name.")

Shape of dataset: (30000, 25)

Column names:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']

Data types of each column:
ID                              int64
LIMIT_BAL                     float64
SEX                             int64
EDUCATION                       int64
MARRIAGE                        int64
AGE                             int64
PAY_0                           int64
PAY_2                           int64
PAY_3                           int64
PAY_4                           int64
PAY_5                           int64
PAY_6                           int64
BILL_AMT1                     float64
BILL_AMT2                     float64
BILL_AMT3                     float64
BILL_AMT4                     float64
BILL_AMT5                 

# Traditional ML Models

Random Forest

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# 1. Define features and target
# -----------------------------
target_col = "default.payment.next.month"

# Drop ID if present, since it is just an identifier and not a useful predictive feature
drop_cols = [target_col]
if "ID" in df.columns:
    drop_cols.append("ID")

X = df.drop(columns=drop_cols)
y = df[target_col]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# -----------------------------------------
# 2. Split data into train/test sets (75/25)
# -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y   # keeps class distribution similar in train and test
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# -------------------------
# 3. Train Random Forest
# -------------------------
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("\nRandom Forest model trained successfully.")

# -------------------------
# 4. Make predictions
# -------------------------
y_pred = rf_model.predict(X_test)

# -------------------------
# 5. Evaluate the model
# -------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\nRandom Forest Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Feature matrix shape: (30000, 23)
Target vector shape: (30000,)

Training set shape: (22500, 23) (22500,)
Testing set shape: (7500, 23) (7500,)

Random Forest model trained successfully.

Random Forest Accuracy: 0.8138666666666666

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      5841
           1       0.65      0.35      0.45      1659

    accuracy                           0.81      7500
   macro avg       0.74      0.65      0.67      7500
weighted avg       0.79      0.81      0.79      7500

Confusion Matrix:
[[5530  311]
 [1085  574]]


XGBoost

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

# -----------------------------
# 1. Define features and target
# -----------------------------
target_col = "default.payment.next.month"

drop_cols = [target_col]
if "ID" in df.columns:
    drop_cols.append("ID")  # remove identifier column if present

X = df.drop(columns=drop_cols)
y = df[target_col]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# -----------------------------------------
# 2. Split data into train/test sets (75/25)
# -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# -------------------------
# 3. Train XGBoost model
# -------------------------
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("\nXGBoost model trained successfully.")

# -------------------------
# 4. Make predictions
# -------------------------
y_pred = xgb_model.predict(X_test)

# -------------------------
# 5. Evaluate the model
# -------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\nXGBoost Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Feature matrix shape: (30000, 23)
Target vector shape: (30000,)

Training set shape: (22500, 23) (22500,)
Testing set shape: (7500, 23) (7500,)

XGBoost model trained successfully.

XGBoost Accuracy: 0.8178666666666666

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      5841
           1       0.66      0.36      0.47      1659

    accuracy                           0.82      7500
   macro avg       0.75      0.66      0.68      7500
weighted avg       0.80      0.82      0.80      7500

Confusion Matrix:
[[5529  312]
 [1054  605]]


# TabPFN (Tabular Prior-Data Fitted Network)

In [7]:
## TabPFN Installation optimized for Google Colab
# Install the TabPFN Client library
!uv pip install tabpfn-client

# Install TabPFN extensions for additional functionalities
!uv pip install tabpfn-extensions[all]

Using Python 3.12.13 environment at: /usr
Resolved 33 packages in 929ms
Prepared 5 packages in 193ms
Uninstalled 1 package in 16ms
Installed 5 packages in 29ms
 + backoff==2.2.1
 + password-strength==0.0.3.post2
 + sseclient-py==1.8.0
 + tabpfn-client==0.2.8
 - tqdm==4.67.3
 + tqdm==4.67.1
Using Python 3.12.13 environment at: /usr
Resolved 93 packages in 2.87s
Prepared 20 packages in 1.13s
Uninstalled 1 package in 9ms
Installed 20 packages in 230ms
 + autogluon-common==1.4.0
 + autogluon-core==1.4.0
 + autogluon-features==1.4.0
 + autogluon-tabular==1.4.0
 + boto3==1.42.73
 + botocore==1.42.73
 + ecos==2.0.14
 + eval-type-backport==0.3.1
 + galois==0.4.10
 + jmespath==1.1.0
 + nvidia-ml-py==13.595.45
 + posthog==7.9.12
 - requests==2.32.4
 + requests==2.32.5
 + s3transfer==0.16.0
 + scikit-survival==0.26.0
 + shapiq==1.4.1
 + sparse-transform==0.2.1
 + tabpfn==6.4.1
 + tabpfn-common-utils==0.2.18
 + tabpfn-extensions==0.2.2


In [8]:
# Simple import for TabPFN
from tabpfn_client import TabPFNClassifier

# Now you can use it like any other sklearn classifier
# model = TabPFNClassifier()
print("TabPFNClassifier imported successfully.")

TabPFNClassifier imported successfully.


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from tabpfn_client import TabPFNClassifier

# -----------------------------
# 1. Define features and target
# -----------------------------
target_col = "default.payment.next.month"

drop_cols = [target_col]
if "ID" in df.columns:
    drop_cols.append("ID")  # drop identifier column if present

X = df.drop(columns=drop_cols).copy()
y = df[target_col].copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# --------------------------------------------------------
# Optional but recommended: verify data is suitable first
# --------------------------------------------------------
print("\nMissing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())
print("Target unique values:", sorted(y.unique()))

# -----------------------------------------
# 2. Split data into train/test sets (75/25)
# -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# -------------------------
# 3. Train TabPFNClassifier
# -------------------------
tabpfn_model = TabPFNClassifier()

tabpfn_model.fit(X_train, y_train)

print("\nTabPFN model trained successfully.")

# -------------------------
# 4. Make predictions
# -------------------------
# Predicted probabilities for both classes
y_pred_proba = tabpfn_model.predict_proba(X_test)

# Predicted class labels
y_pred = tabpfn_model.predict(X_test)

print("\nPrediction probability array shape:", y_pred_proba.shape)

# For binary classification, use probability of the positive class (class 1)
y_pred_proba_class1 = y_pred_proba[:, 1]

# -------------------------
# 5. Evaluate the model
# -------------------------
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba_class1)

print("\nTabPFN Accuracy:", accuracy)
print("TabPFN ROC AUC:", roc_auc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Feature matrix shape: (30000, 23)
Target vector shape: (30000,)

Missing values in X: 0
Missing values in y: 0
Target unique values: [np.int64(0), np.int64(1)]

Training set shape: (22500, 23) (22500,)
Testing set shape: (7500, 23) (7500,)


########  ########   ###  #########  #########       ###         #####     ########  ########
     ###        ##   ###  ###   ###        ###       ###        ###  ###   ##   ###  ###     
########  #######    ###  ###   ###  #######         ###        ########   ######    ########
###       ###   ##   ###  ###   ###  ###   ###       ###        ###  ###   ##   ###       ###
###       ###   ##   ###  #########  ###   ###       ########   ###  ###   ########  ########                      

Thanks for being part of the journey

TabPFN is under active development, please help us improve and report any bugs/ideas you find.

Report issues: https://github.com/priorlabs/tabpfn-client/issues

...

Output()

Login successful!


TabPFN model trained successfully.


Processing: 100%|██████████| [00:05<00:00]
Processing: 100%|██████████| [00:04<00:00]



Prediction probability array shape: (7500, 2)

TabPFN Accuracy: 0.8188
TabPFN ROC AUC: 0.7847345348954445

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      5841
           1       0.66      0.36      0.47      1659

    accuracy                           0.82      7500
   macro avg       0.75      0.66      0.68      7500
weighted avg       0.80      0.82      0.80      7500

Confusion Matrix:
[[5536  305]
 [1054  605]]


# Review

TabPFN has the best overall accuracy at 0.8188, slightly above XGBoost at 0.8179 and Random Forest at 0.8139, meaning its overall classification performance on the held-out test set is marginally strongest, but the improvement over XGBoost is extremely small and likely not practically significant on its own.

For the minority class, which is usually the more important class here because it represents default risk, TabPFN is essentially tied with XGBoost. Both models have precision of 0.66, recall of 0.36, and F1-score of 0.47 for class 1. Their confusion matrices are also nearly identical. This suggests TabPFN does not materially improve detection of defaulters compared with XGBoost, though both outperform Random Forest slightly, especially in recall and F1 for class 1.

Compared with Random Forest, TabPFN is clearly a bit better. Random Forest has lower class-1 recall at 0.35 and lower F1 at 0.45, meaning it misses slightly more defaults. So TabPFN shows a modest gain over Random Forest in identifying risky customers.

The ROC—AUC of TabPFN is 0.7847, which indicates reasonably good ranking ability for a binary classification problem. Since ROC AUC was only reported for TabPFN, it is not possible to make a full ranking-performance comparison against Random Forest and XGBoost from the numbers shown. But taken together with the classification metrics, TabPFN appears to be strong and well-calibrated enough to match the best traditional model here.